# Auliya E3 — final bounded mechanism diagnosis

No training. Attach extracted auliya-robusta-sni and the same full G1/G2 notebook output that succeeded in E1. Both historical last.pt checkpoints are hash-verified. Enable GPU and Internet. Run All. E3 uses the native mask-aware assignment on source training data, mild view transformations and candidate alternatives on development evaluation. No A access.


In [ ]:
# Settings
SCIENTIFIC_CODE_COMMIT = "8fe1d733db30bde0389e5ce358171631518e7096"
RUN_E1 = True  # require both exact historical checkpoints for E3
RUN_HAAR = False
DEVICE = "0"
DEVELOPMENT_OVERRIDE = ""  # optional exact mounted development directory
METADATA_OVERRIDE = ""     # optional mounted official metadata ZIP
RGB_REPORT_ZIP = ""        # optional explicit original report ZIP
RGB_B_CHECKPOINT = ""      # optional exact B last.pt path
RGB_C_CHECKPOINT = ""      # optional exact C last.pt path
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
from urllib.request import urlopen

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
PROJECT = WORK / "auliya-gc-e3-v1"
REPO = WORK / "coffee-bean-detection-gc-e3"
DATA = WORK / "auliya-gc-e3-data"
DOWNLOAD = WORK / "auliya-gc-e3-assets"
assert INPUT.is_dir() and WORK.is_dir(), "Run on Kaggle"
DOWNLOAD.mkdir(exist_ok=True)
DEV_SHA = "23d653ca2e98c185a76bb7b2095d6afc6401876cbe48fd219cb42bdc7ecbd360"
META_SHA = "798bb53b0cfad35dfcfbcc463fbcb79ff4af61f9807803d6bcc65d37f7c8feaa"
MODEL_SHA = "3da1d83e31caec96f9300eb4064f4f62882c133c7c264d63dfe61a7c197837a4"

def digest(p):
    h = hashlib.sha256()
    with Path(p).open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

if DEVELOPMENT_OVERRIDE:
    devs = [Path(DEVELOPMENT_OVERRIDE).resolve()]
else:
    devs = sorted(p.parent for p in INPUT.rglob("development_contract.json")
                  if p.parent.name == "development" and digest(p) == DEV_SHA)
if len(devs) != 1:
    raise RuntimeError(f"Expected exactly one verified development folder; found {devs}")
DEVELOPMENT = devs[0]
if not DEVELOPMENT.resolve().is_relative_to(INPUT.resolve()) or digest(DEVELOPMENT/"development_contract.json") != DEV_SHA:
    raise RuntimeError("Invalid mounted development contract")
print("Development:", DEVELOPMENT, flush=True)

prior = sorted(p for p in INPUT.rglob(PROJECT.name) if p.is_dir() and (p/"logs").is_dir())
if len(prior)>1:
    raise RuntimeError("Attach at most one previous Notebook Output for resume")
if prior and not PROJECT.exists():
    shutil.copytree(prior[0],PROJECT)
elif prior:
    print("Using current working output; not overwriting with prior")
PROJECT.mkdir(exist_ok=True)
(PROJECT/"logs").mkdir(exist_ok=True)

def fetch_verified(url, destination, expected):
    if destination.is_file() and digest(destination)==expected:
        return destination
    tmp = destination.with_suffix(".part")
    try:
        with urlopen(url,timeout=120) as source, tmp.open("wb") as sink:
            shutil.copyfileobj(source,sink)
        if digest(tmp)!=expected:
            raise RuntimeError(f"SHA mismatch: {destination}")
        tmp.replace(destination)
    finally:
        tmp.unlink(missing_ok=True)
    return destination

def run(cmd,log=None,cwd=None):
    cmd=list(map(str,cmd))
    print("$ "+" ".join(cmd),flush=True)
    if log is None:
        subprocess.run(cmd,check=True,cwd=cwd)
        return
    with Path(log).open("a",encoding="utf-8") as stream:
        with subprocess.Popen(cmd,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
            for line in p.stdout:
                print(line,end="",flush=True)
                stream.write(line)
                stream.flush()
            if p.wait()!=0:
                raise RuntimeError(f"Command failed; see {log}")

if METADATA_OVERRIDE:
    METADATA=Path(METADATA_OVERRIDE).resolve()
    if not METADATA.is_relative_to(INPUT.resolve()) or digest(METADATA)!=META_SHA:
        raise RuntimeError("Official metadata override invalid")
else:
    METADATA=fetch_verified("https://zenodo.org/api/records/24346584/files/splits_metadata_results_scripts.zip/content",DOWNLOAD/"metadata.zip",META_SHA)
print("Verified metadata.")

# Resolve reports and checkpoints independently, using exact original hashes.
from zipfile import ZipFile, BadZipFile
EXPECTED = {"B":"38385effe9335bf5ad23fee9913f19f5373494da1c20bdb578e1138ae52719fc",
            "C":"41ffe7f52503d64ca4a008dd477e0575aa52604c8274f5c7005f1fb0a0723642"}
assert RUN_E1 or RUN_HAAR
RGB_RUNS = PROJECT/"verified_rgb_references"
RGB_RUNS.mkdir(exist_ok=True)
roots = (INPUT, WORK)
found_reports = {g:[] for g in "BC"}
found_weights = {g:[] for g in "BC"}
def register_report(raw, origin):
    try:
        report = json.loads(raw)
        group = report["run_contract"]["group"]
        if (group in EXPECTED and report["checkpoint_sha256"] == EXPECTED[group]
                and report["completed_epochs"] == 80
                and report["test_A_images_accessed"] is False
                and report["run_contract"]["protocol"] == "auliya-controlled-generalization-v1"):
            found_reports[group].append((report, str(origin)))
    except (ValueError, KeyError, TypeError, AttributeError):
        pass

report_files = sorted({p for root in roots for p in root.rglob("validation_result.json") if p.is_file()})
for p in report_files:
    register_report(p.read_text(encoding="utf-8"), p)
report_zips = [Path(RGB_REPORT_ZIP)] if RGB_REPORT_ZIP else sorted({
    p for root in roots for p in root.rglob("auliya-controlled-generalization-report.zip") if p.is_file()
})
for p in report_zips:
    try:
        with ZipFile(p) as z:
            for entry in z.infolist():
                if entry.filename.endswith("validation_result.json") and entry.file_size < 2_000_000:
                    register_report(z.read(entry), f"{p}:{entry.filename}")
    except (OSError, BadZipFile) as exc:
        print("Cannot read report ZIP:", p, exc)
for group in "BC":
    matches = found_reports[group]
    if matches:
        report = matches[0][0]
        if any(other != report for other, _ in matches):
            raise RuntimeError(f"Conflicting original RGB reports for group {group}")
        target = RGB_RUNS/group/"validation_result.json"
        target.parent.mkdir(exist_ok=True)
        target.write_text(json.dumps(report,indent=2),encoding="utf-8")
        print(f"RGB {group} report:", matches[0][1])
    else:
        print(f"RGB {group} report: MISSING")
if RUN_E1:
    # Accept renamed checkpoints and different mounted output layouts.
    # No best.pt substitution or new checkpoint accepted unless its bytes match.
    paths = sorted({p for root in roots for p in root.rglob("*.pt") if p.is_file()})
    overrides = {"B": RGB_B_CHECKPOINT, "C": RGB_C_CHECKPOINT}
    digests = {}
    for group in "BC":
        pool = [Path(overrides[group])] if overrides[group] else paths
        for p in pool:
            if not p.is_file():
                print("Checkpoint path does not exist:",p)
                continue
            rp = p.resolve()
            if rp not in digests:
                digests[rp] = digest(p)
            if digests[rp] == EXPECTED[group]:
                found_weights[group].append(p)
        if found_weights[group]:
            print(f"RGB {group} checkpoint VERIFIED:",found_weights[group][0])
        else:
            print(f"RGB {group} checkpoint: MISSING (expected SHA256 {EXPECTED[group]})")
    print("Checkpoint inventory (path, bytes, SHA256):")
    for p,sha in list(digests.items())[:100]:
        print(p,p.stat().st_size,sha)
missing = [f"report {g}" for g in "BC" if not found_reports[g]]
if RUN_E1:
    missing += [f"checkpoint {g}" for g in "BC" if not found_weights[g]]
if missing:
    print("Mounted input roots:",sorted(str(p) for p in INPUT.iterdir()))
    print("Discovered report JSON paths:",[str(p) for p in report_files[:100]])
    raise RuntimeError("Missing exact historical artifacts: "+", ".join(missing)+
        ". Attach full saved G1/G2 output, or original report ZIP plus both last.pt files. "
        "Report-only ZIP cannot supply model weights. See inventory above.")
if RUN_E1:
    CHECKPOINT_B = found_weights["B"][0]
    CHECKPOINT_C = found_weights["C"][0]
print("Verified RGB references:",RGB_RUNS)


In [ ]:
# Pinned scientific code, no package import from bundled Kaggle data.
if not REPO.exists():
    run(["git","clone","--quiet","--no-checkout","https://github.com/ediprin/coffee-bean-detection.git",REPO])
if subprocess.run(["git","-C",str(REPO),"cat-file","-e",SCIENTIFIC_CODE_COMMIT+"^{commit}"],capture_output=True).returncode!=0:
    run(["git","-C",REPO,"fetch","--quiet","origin","codex/auliya-controlled-generalization"])
run(["git","-C",REPO,"checkout","--quiet","--detach",SCIENTIFIC_CODE_COMMIT])
assert subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()==SCIENTIFIC_CODE_COMMIT
os.environ["PYTHONPATH"]=str(REPO/"src")+os.pathsep+os.environ.get("PYTHONPATH","")
os.environ["PYTHONUNBUFFERED"]="1"
os.environ["OMP_NUM_THREADS"]="2"
# Official fold intentionally uses newer Ultralytics than the pilot.
run([sys.executable,"-m","pip","install","-q","ultralytics==8.4.210","scikit-learn","pytest"])
# Use subprocess to avoid stale imports from earlier Kaggle cells.
runtime=json.loads(subprocess.check_output([sys.executable,"-c",
    "import json,torch,ultralytics,numpy,cv2,sklearn,platform; print(json.dumps(dict(torch=str(torch.__version__),ultralytics=ultralytics.__version__,numpy=numpy.__version__,opencv=cv2.__version__,sklearn=sklearn.__version__,python=platform.python_version(),gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)))"],text=True).strip().splitlines()[-1])
assert runtime["ultralytics"]=="8.4.210",runtime
if (RUN_E1 or RUN_HAAR) and runtime["gpu"] is None:
    raise RuntimeError("Enable Kaggle GPU")
runtime["code_commit"]=SCIENTIFIC_CODE_COMMIT
marker=PROJECT/"environment.json"
if marker.exists() and json.loads(marker.read_text())!=runtime:
    raise RuntimeError("Runtime changed; do not resume in a different environment")
marker.write_text(json.dumps(runtime,indent=2))
print(runtime)
run([sys.executable,"-m","pip","freeze"],log=PROJECT/"logs/pip-freeze.txt")
run([sys.executable,"-m","pytest","-q","tests/test_auliya_official_fold.py","tests/test_auliya_representation_audit.py","tests/test_auliya_controlled_generalization.py","tests/test_auliya_gc_error_audit.py","tests/test_auliya_gc_mechanism_audit.py"],cwd=REPO,log=PROJECT/"logs/tests.log")


In [ ]:
# E3: frozen inference and native mask-aware assignment; no optimizer or training.
run([sys.executable,"-u","-m","coffee_detector.analysis.auliya_gc_mechanism_audit",
     "--development",DEVELOPMENT,"--metadata",METADATA,"--data",DATA,
     "--checkpoint-b",CHECKPOINT_B,"--checkpoint-c",CHECKPOINT_C,
     "--output",PROJECT/"E3","--device","cuda:0"],cwd=REPO,log=PROJECT/"logs/E3.log")
report=json.loads((PROJECT/"E3/report.json").read_text())
assert report["protocol"]=="auliya-gc-e3-v1" and report["training_executed"] is False and report["test_A_images_accessed"] is False
assert report["assignment_records"]==3752 and report["evaluation_records"]==3960
print("E1 replication:",report["e1_original_replication"])
for k,r in report["results"]["ranking"].items():
    if k.endswith("/original/clean18") or k.endswith("/original/hard5"):
        print(k,r)


In [ ]:
# Print core supervision and stability summaries.
for k,r in report["results"]["assignment"].items():
    if k.endswith("/clean18") or k.endswith("/hard5"):
        print("Assignment",k,r)
for k,r in report["results"]["stability"].items():
    if k.endswith("/hard5/score"):
        print("Stability",k,r)


In [ ]:
# Report only; no checkpoint duplication.
from zipfile import ZipFile, ZIP_DEFLATED
archive=WORK/"auliya-gc-e3-report.zip"
with ZipFile(archive,"w",compression=ZIP_DEFLATED) as z:
    for p in sorted(PROJECT.rglob("*")):
        if p.is_file() and p.suffix.lower() not in {".pt",".part",".tmp"}:
            z.write(p,p.relative_to(PROJECT))
print("Upload report:",archive)
